# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. By referencing all entities using their `@id` fields, the workflow ensures clarity and reproducibility for programmatic data interaction.

### Dataset Source
The dataset is described via a Croissant schema accessible at the following URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant pandas

## 1. Data Loading
Load the metadata and records from the dataset using `mlcroissant`.

_References:_
- [mlcroissant documentation](https://mlcommons.github.io/croissant/sdk/mlcroissant.html#mlcroissant.Dataset)


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
# Access metadata as object attributes
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Schema version: {dataset.metadata.version}")

## 2. Data Overview
Let's review available record sets in the dataset using their `@id`. We also examine the fields within one record set and their corresponding `@id`, since all references must use `@id` fields per Croissant convention.

In [ ]:
# List all record set @id's available in the Croissant schema
record_sets = dataset.metadata.record_sets
print("Available Record Sets and their @ids:")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs['name']} (type: {rs.get('@type', 'RecordSet')})")

In [ ]:
# Example: Show the fields (@id and name) for the first record set
if len(record_sets) == 0:
    print("No record sets defined in the metadata (schema may need to be updated or checked).")
else:
    # Select the first record set for exploration
    record_set0 = record_sets[0]
    print(f"Fields for RecordSet {record_set0['@id']}:")
    for field in record_set0['fields']:
        field_id = field['@id']
        field_name = field.get('name', '')
        field_type = field.get('@type', '')
        print(f"  - {field_id}: {field_name} (type: {field_type})")

## 3. Data Extraction
Let's programmatically extract the data for each record set by referencing all `@id` fields, and load them into pandas DataFrames for analysis.

In [ ]:
# List of record set @ids (referenced dynamically from metadata)
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

# Load each record set using its @id
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows in DataFrame for record set: {record_set_id}")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

In [ ]:
# Display column names for the first record set
if record_set_ids:
    main_rs = record_set_ids[0]
    print(f"Columns in record set '{main_rs}':")
    print(dataframes[main_rs].columns.tolist())
    display(dataframes[main_rs].head())
else:
    print('No record sets available.')

## 4. Exploratory Data Analysis (EDA)
We demonstrate filtering, normalizing, and grouping data based on numerical and categorical fields.

You may adjust the field `@id`s used below as needed, referencing the printout from above. If the field types are not known, inspect the DataFrame or schema for numeric/categorical candidates.

In [ ]:
# Choose a numeric field and a group field from the first record set
main_df = dataframes[main_rs]

# List potential numeric columns for analysis
numeric_fields = main_df.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric fields detected: {numeric_fields}")

if not numeric_fields:
    print('No numeric fields detected; cannot perform numeric EDA.')
else:
    numeric_field = numeric_fields[0]  # e.g., '@cr:protein_coverage' or similar
    print(f\nUsing numeric field: {numeric_field}")

    threshold = main_df[numeric_field].mean() if main_df[numeric_field].notnull().any() else 0
    filtered_df = main_df[main_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize field
    filtered_df[numeric_field + '_normalized'] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())
    
    # Try to find a categorical field for grouping
    candidate_group_fields = main_df.select_dtypes(include=['object']).columns.tolist()
    group_field = candidate_group_fields[0] if candidate_group_fields else None

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields. Below, we show an example histogram and boxplot based on one of the available numeric fields. Customize further based on the dataset contents and EDA findings.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # If there's a group field, show boxplot
    if group_field:
        plt.figure(figsize=(12, 4))
        sns.boxplot(data=main_df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45, ha='right')
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to programmatically explore a Croissant-compliant dataset using the `mlcroissant` library. All interactions referenced entities by their `@id` fields for consistent, robust data access and manipulation.

We covered:
- Loading dataset metadata and inspecting available record sets (`@id`).
- Extracting records and examining field structure.
- Filtering, normalizing, and grouping data by field `@id`.
- Plotting basic distributions and group-wise summaries.

Further exploration can extend these analyses for more in-depth biological or methodological insights.